<a href="https://colab.research.google.com/github/6pb4wnww4g-beep/Jason/blob/main/Jason_PS_LAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- THE ODDS API TEST FOR JASON 94.0 ---
# Formål:
# - Teste hva vi faktisk får ut av The Odds API for Premier League
# - Skrive rå og ryddet market-data til Google Sheets
#
# Før kjøring:
# 1. Lim inn din egen API-key under API_KEY
# 2. Kjør i Colab
#
# Skriver til ark:
# - OddsAPI_Raw_Matches
# - OddsAPI_Market_Check
# - OddsAPI_Bookmaker_Detail
#
# Viktig:
# Dette er IKKE Jason 94.0.
# Dette er kun en datatest før vi bygger marketdelen inn i Jason.

import requests
import pandas as pd
import numpy as np
import time
from google.colab import auth
import gspread
from google.auth import default

API_KEY = "1febfae459fe024b33c82de75598bd9f"
SPREADSHEET_NAME = "Jason Development"
SPORT_KEY = "soccer_epl"

# h2h = 1X2
# totals = Over/Under
# btts og team_totals testes hvis API/plan støtter dem
MARKETS_TO_TEST = "h2h,totals,btts,team_totals"

REGIONS = "uk,eu"
ODDS_FORMAT = "decimal"
DATE_FORMAT = "iso"
BASE_URL = "https://api.the-odds-api.com/v4"

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.open(SPREADSHEET_NAME)


def update_worksheet(sheet_name, dataframe):
    print(f"Oppdaterer {sheet_name}...")
    df_clean = dataframe.replace([np.inf, -np.inf], np.nan).fillna("")
    try:
        ws = sh.worksheet(sheet_name)
        ws.clear()
    except Exception:
        ws = sh.add_worksheet(title=sheet_name, rows="2000", cols="80")

    ws.resize(
        rows=max(1000, len(df_clean) + 20),
        cols=max(30, len(df_clean.columns) + 5)
    )
    ws.update([df_clean.columns.tolist()] + df_clean.values.tolist())
    time.sleep(2)


def get_remaining_headers(response):
    return {
        "x-requests-remaining": response.headers.get("x-requests-remaining", ""),
        "x-requests-used": response.headers.get("x-requests-used", ""),
        "x-requests-last": response.headers.get("x-requests-last", "")
    }


def fetch_available_sports():
    url = f"{BASE_URL}/sports/"
    params = {"apiKey": API_KEY}
    response = requests.get(url, params=params)
    print("Sports status:", response.status_code)
    print("Quota:", get_remaining_headers(response))

    if response.status_code != 200:
        print(response.text[:1000])
        return pd.DataFrame()

    return pd.DataFrame(response.json())


def fetch_odds(markets):
    url = f"{BASE_URL}/sports/{SPORT_KEY}/odds/"
    params = {
        "apiKey": API_KEY,
        "regions": REGIONS,
        "markets": markets,
        "oddsFormat": ODDS_FORMAT,
        "dateFormat": DATE_FORMAT
    }
    response = requests.get(url, params=params)
    print(f"Odds status for markets [{markets}]:", response.status_code)
    print("Quota:", get_remaining_headers(response))

    if response.status_code != 200:
        print("Feil fra The Odds API:")
        print(response.text[:2000])
        return []

    return response.json()


def parse_matches(raw_matches):
    raw_rows = []
    market_rows = []
    bookmaker_rows = []

    for match in raw_matches:
        match_id = match.get("id", "")
        commence_time = match.get("commence_time", "")
        home_team = match.get("home_team", "")
        away_team = match.get("away_team", "")
        bookmakers = match.get("bookmakers", [])

        raw_rows.append({
            "match_id": match_id,
            "sport_key": match.get("sport_key", ""),
            "sport_title": match.get("sport_title", ""),
            "commence_time": commence_time,
            "home_team": home_team,
            "away_team": away_team,
            "bookmakers_count": len(bookmakers),
            "bookmakers": ", ".join([b.get("title", "") for b in bookmakers])
        })

        h2h_home_prices, h2h_draw_prices, h2h_away_prices = [], [], []
        totals_over_prices, totals_under_prices, totals_points = [], [], []
        btts_yes_prices, btts_no_prices = [], []
        team_total_rows_found = 0
        markets_found = set()

        for bookmaker in bookmakers:
            bm_key = bookmaker.get("key", "")
            bm_title = bookmaker.get("title", "")
            bm_last_update = bookmaker.get("last_update", "")

            for market in bookmaker.get("markets", []):
                market_key = market.get("key", "")
                markets_found.add(market_key)
                outcomes = market.get("outcomes", [])

                for outcome in outcomes:
                    bookmaker_rows.append({
                        "match_id": match_id,
                        "commence_time": commence_time,
                        "home_team": home_team,
                        "away_team": away_team,
                        "bookmaker_key": bm_key,
                        "bookmaker": bm_title,
                        "last_update": bm_last_update,
                        "market": market_key,
                        "outcome_name": outcome.get("name", ""),
                        "price": outcome.get("price", ""),
                        "point": outcome.get("point", ""),
                        "description": outcome.get("description", "")
                    })

                if market_key == "h2h":
                    for outcome in outcomes:
                        n = outcome.get("name", "")
                        p = outcome.get("price", None)
                        if p is None:
                            continue
                        if n == home_team:
                            h2h_home_prices.append(float(p))
                        elif n == away_team:
                            h2h_away_prices.append(float(p))
                        elif n.lower() == "draw":
                            h2h_draw_prices.append(float(p))

                elif market_key == "totals":
                    for outcome in outcomes:
                        n = outcome.get("name", "").lower()
                        p = outcome.get("price", None)
                        point = outcome.get("point", None)
                        if p is None:
                            continue
                        if point is not None:
                            totals_points.append(point)
                        if n == "over":
                            totals_over_prices.append(float(p))
                        elif n == "under":
                            totals_under_prices.append(float(p))

                elif market_key in ["btts", "both_teams_to_score"]:
                    for outcome in outcomes:
                        n = outcome.get("name", "").lower()
                        p = outcome.get("price", None)
                        if p is None:
                            continue
                        if n in ["yes", "true"]:
                            btts_yes_prices.append(float(p))
                        elif n in ["no", "false"]:
                            btts_no_prices.append(float(p))

                elif market_key in ["team_totals", "team_total"]:
                    team_total_rows_found += len(outcomes)

        def avg(xs):
            return round(sum(xs) / len(xs), 3) if xs else ""

        def unique_points(xs):
            clean = sorted(list(set([x for x in xs if x not in [None, ""]])))
            return ", ".join([str(x) for x in clean])

        market_rows.append({
            "match_id": match_id,
            "commence_time": commence_time,
            "home_team": home_team,
            "away_team": away_team,
            "bookmakers_count": len(bookmakers),
            "markets_found": ", ".join(sorted(markets_found)),
            "avg_home_odds": avg(h2h_home_prices),
            "avg_draw_odds": avg(h2h_draw_prices),
            "avg_away_odds": avg(h2h_away_prices),
            "avg_over_odds": avg(totals_over_prices),
            "avg_under_odds": avg(totals_under_prices),
            "totals_lines_found": unique_points(totals_points),
            "avg_btts_yes": avg(btts_yes_prices),
            "avg_btts_no": avg(btts_no_prices),
            "team_totals_rows_found": team_total_rows_found,
            "h2h_books": len(h2h_home_prices),
            "totals_books": len(totals_over_prices),
            "btts_books": len(btts_yes_prices)
        })

    return pd.DataFrame(raw_rows), pd.DataFrame(market_rows), pd.DataFrame(bookmaker_rows)


if API_KEY == "LIM_INN_DIN_API_KEY_HER":
    raise ValueError("Lim inn API-key i API_KEY før du kjører scriptet.")

print("Tester tilgjengelige sports...")
sports_df = fetch_available_sports()
if not sports_df.empty:
    epl_rows = sports_df[
        sports_df.astype(str).apply(
            lambda row: row.str.contains("epl|premier", case=False, regex=True).any(),
            axis=1
        )
    ]
    print("Mulige EPL-rader i sports-listen:")
    print(epl_rows.head(20))

print("Henter odds...")
raw_matches = fetch_odds(MARKETS_TO_TEST)

# Fallback hvis noen markets ikke er støttet på gratisplanen:
if not raw_matches:
    print("Prøver fallback med h2h,totals...")
    raw_matches = fetch_odds("h2h,totals")

if not raw_matches:
    print("Prøver fallback med kun h2h...")
    raw_matches = fetch_odds("h2h")

print(f"Antall kamper hentet: {len(raw_matches)}")

raw_df, market_df, detail_df = parse_matches(raw_matches)

update_worksheet("OddsAPI_Raw_Matches", raw_df)
update_worksheet("OddsAPI_Market_Check", market_df)
update_worksheet("OddsAPI_Bookmaker_Detail", detail_df)

print("Ferdig.")
print("Sjekk arkene:")
print("- OddsAPI_Raw_Matches")
print("- OddsAPI_Market_Check")
print("- OddsAPI_Bookmaker_Detail")

if not market_df.empty:
    print("")
    print("Kort oppsummering:")
    print("Markets funnet:")
    print(market_df["markets_found"].value_counts().head(20))
    print("")
    print("Kamper med totals:")
    print((market_df["totals_books"] > 0).sum())
    print("")
    print("Kamper med BTTS:")
    print((market_df["btts_books"] > 0).sum())
    print("")
    print("Kamper med team totals:")
    print((market_df["team_totals_rows_found"] > 0).sum())

Tester tilgjengelige sports...
Sports status: 200
Quota: {'x-requests-remaining': '500', 'x-requests-used': '0', 'x-requests-last': '0'}
Mulige EPL-rader i sports-listen:
                         key     group              title  \
22              lacrosse_pll  Lacrosse                PLL   
31                soccer_epl    Soccer                EPL   
37  soccer_league_of_ireland    Soccer  League of Ireland   

                           description  active  has_outrights  
22             Premier Lacrosse League    True          False  
31              English Premier League    True          False  
37  Airtricity League Premier Division    True          False  
Henter odds...
Odds status for markets [h2h,totals,btts,team_totals]: 422
Quota: {'x-requests-remaining': '500', 'x-requests-used': '0', 'x-requests-last': '0'}
Feil fra The Odds API:
{"message":"Markets not supported by this endpoint: btts, team_totals","error_code":"INVALID_MARKET","details_url":"https://the-odds-api.com/liv